In [23]:
# dependencies
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset


# Rishis notebook

In [9]:
df = pd.read_csv(
    "LD2011_2014.txt",
    sep=";",              # delimiter
    decimal=",",          # VERY IMPORTANT
    parse_dates=[0],      # first column = datetime
    quotechar='"'         # handles quotes
)
df.head()
df.rename(columns={df.columns[0]: "datetime"}, inplace=True)
df.set_index("datetime", inplace=True)
df.isna().sum()[0]==1
df.columns
df["total_load"] = df.sum(axis=1)
df_hourly = df["total_load"].resample("H").sum().to_frame()
df_hourly.head()
# If df_hourly is a Series, convert it to a DataFrame first
if isinstance(df_hourly, pd.Series):
    df_hourly = df_hourly.to_frame(name="total_load")

# Make sure index is datetime
df_hourly.index = pd.to_datetime(df_hourly.index)

# Create a copy so original stays untouched
df_features = df_hourly.copy()

# Basic calendar/time features
df_features["hour_of_day"] = df_features.index.hour
df_features["day_of_week"] = df_features.index.dayofweek   # Monday=0, Sunday=6
df_features["is_weekend"] = (df_features["day_of_week"] >= 5).astype(int)
df_features["month"] = df_features.index.month

# Optional cyclical encoding
df_features["hour_sin"] = np.sin(2 * np.pi * df_features["hour_of_day"] / 24)
df_features["hour_cos"] = np.cos(2 * np.pi * df_features["hour_of_day"] / 24)

df_features["dayofweek_sin"] = np.sin(2 * np.pi * df_features["day_of_week"] / 7)
df_features["dayofweek_cos"] = np.cos(2 * np.pi * df_features["day_of_week"] / 7)

df_features["month_sin"] = np.sin(2 * np.pi * (df_features["month"] - 1) / 12)
df_features["month_cos"] = np.cos(2 * np.pi * (df_features["month"] - 1) / 12)

print(df_features.head())
print(df_features.columns)

                        total_load  hour_of_day  day_of_week  is_weekend  \
datetime                                                                   
2011-01-01 00:00:00  207058.270272            0            5           1   
2011-01-01 01:00:00  265378.510747            1            5           1   
2011-01-01 02:00:00  263924.219533            2            5           1   
2011-01-01 03:00:00  266306.134264            3            5           1   
2011-01-01 04:00:00  259854.210701            4            5           1   

                     month  hour_sin  hour_cos  dayofweek_sin  dayofweek_cos  \
datetime                                                                       
2011-01-01 00:00:00      1  0.000000  1.000000      -0.974928      -0.222521   
2011-01-01 01:00:00      1  0.258819  0.965926      -0.974928      -0.222521   
2011-01-01 02:00:00      1  0.500000  0.866025      -0.974928      -0.222521   
2011-01-01 03:00:00      1  0.707107  0.707107      -0.974928      

/tmp/ipykernel_18111/1167452158.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df.isna().sum()[0]==1
/tmp/ipykernel_18111/1167452158.py:14: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df["total_load"].resample("H").sum().to_frame()


# normalizing df

In [15]:
def normalize_and_split(df_features, train_end='2013-12-31', val_start='2014-01-01',
                        val_end='2014-06-30', test_start='2014-07-01'):

    from sklearn.preprocessing import StandardScaler

    # Chronological split
    train = df_features[:train_end].copy()
    val = df_features[val_start:val_end].copy()
    test = df_features[test_start:].copy()

    # Scale total_load using only training data
    scaler = StandardScaler()
    scaler.fit(train[['total_load']])

    train['total_load_scaled'] = scaler.transform(train[['total_load']])
    val['total_load_scaled'] = scaler.transform(val[['total_load']])
    test['total_load_scaled'] = scaler.transform(test[['total_load']])

    # Print summary
    print(f"Train: {train.index.min()} to {train.index.max()} ({len(train)} rows)")
    print(f"Val:   {val.index.min()} to {val.index.max()} ({len(val)} rows)")
    print(f"Test:  {test.index.min()} to {test.index.max()} ({len(test)} rows)")

    return train, val, test, scaler

In [16]:
train, val, test, scaler = normalize_and_split(df_features)

Train: 2011-01-01 00:00:00 to 2013-12-31 23:00:00 (26304 rows)
Val:   2014-01-01 00:00:00 to 2014-06-30 23:00:00 (4344 rows)
Test:  2014-07-01 00:00:00 to 2015-01-01 00:00:00 (4417 rows)


create sequences

In [20]:
def create_sequences(data, input_window=168, forecast_horizon=24):
    feature_cols = ['total_load_scaled', 'hour_sin', 'hour_cos',
                    'dayofweek_sin', 'dayofweek_cos',
                    'month_sin', 'month_cos']

    features = data[feature_cols].values
    targets = data['total_load_scaled'].values

    X, y = [], []
    for i in range(len(features) - input_window - forecast_horizon + 1):
        X.append(features[i : i + input_window])
        y.append(targets[i + input_window : i + input_window + forecast_horizon])

    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)

In [21]:
X_train, y_train = create_sequences(train)
X_val, y_val = create_sequences(val)
X_test, y_test = create_sequences(test)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

X_train: torch.Size([26113, 168, 7]), y_train: torch.Size([26113, 24])
X_val:   torch.Size([4153, 168, 7]),   y_val:   torch.Size([4153, 24])
X_test:  torch.Size([4226, 168, 7]),  y_test:  torch.Size([4226, 24])


Data loader ( batching)

In [24]:


def create_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test, batch_size=64):
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

    # Sanity check
    x_batch, y_batch = next(iter(train_loader))
    print(f"Input batch shape:  {x_batch.shape}")
    print(f"Target batch shape: {y_batch.shape}")

    return train_loader, val_loader, test_loader



In [25]:
train_loader, val_loader, test_loader = create_dataloaders(X_train, y_train, X_val, y_val, X_test, y_test)

Input batch shape:  torch.Size([64, 168, 7])
Target batch shape: torch.Size([64, 24])


persistence baseline( for comparision)